# AI Agent Task: Autonomous Agent & Multi-Agent Simulation

This notebook uses Python to build, run, and analyze a simple autonomous AI agent system. The task focuses on the ReAct reasoning loop, planning, memory, tool calling, and multi-agent collaboration — implemented as lightweight, rule-based simulations so the concepts can run without any external API keys.

## Prompt Used

**Prompt:**

Act as an AI agent developer. Build a simple, beginner-friendly simulation of an autonomous AI agent using Python. Implement a Tool class, a Memory class, and an Agent class that follows the ReAct (Reason + Act) loop to complete a task using available tools.

Extend the simulation to show a Planning Agent that breaks a goal into sub-tasks, and a simple Multi-Agent System with a Manager agent that delegates sub-tasks to specialized Worker agents.

Keep the code beginner-friendly, use clear print statements to show each reasoning step, and explain the results after each section. Keep the task focused on demonstrating agent concepts rather than production-grade infrastructure or real LLM calls.

In [1]:
# Core building blocks: Tool and Memory classes
import random

class Tool:
    """A simple tool an agent can call."""
    def __init__(self, name, description, func):
        self.name = name
        self.description = description
        self.func = func

    def use(self, *args, **kwargs):
        return self.func(*args, **kwargs)


class Memory:
    """A simple short-term memory store for an agent."""
    def __init__(self):
        self.log = []

    def remember(self, entry):
        self.log.append(entry)

    def recall(self, n=5):
        return self.log[-n:]

    def show(self):
        for i, entry in enumerate(self.log, 1):
            print(f"{i}. {entry}")

print("Core Tool and Memory classes defined successfully!")

Core Tool and Memory classes defined successfully!


## 1. Defining Tools

Two simple tools are defined for the agent to use: a **search tool** that looks up facts from a small knowledge base, and a **calculator tool** that evaluates basic math expressions. These stand in for real-world tools such as web search APIs or code execution engines.

In [2]:
# Sample knowledge base the "search" tool will look up from
knowledge_base = {
    "capital of france": "Paris",
    "largest planet": "Jupiter",
    "speed of light": "299,792 km/s",
    "python creator": "Guido van Rossum",
}

def search_tool_func(query):
    key = query.lower().strip()
    return knowledge_base.get(key, "No information found for that query.")

def calculator_tool_func(expression):
    try:
        return eval(expression, {"__builtins__": {}})
    except Exception as e:
        return f"Error evaluating expression: {e}"

search_tool = Tool("search", "Looks up a fact from the knowledge base.", search_tool_func)
calculator_tool = Tool("calculator", "Evaluates a basic math expression.", calculator_tool_func)

tools = {tool.name: tool for tool in [search_tool, calculator_tool]}

print("Available tools:")
for t in tools.values():
    print(f"- {t.name}: {t.description}")

Available tools:
- search: Looks up a fact from the knowledge base.
- calculator: Evaluates a basic math expression.


## 2. The ReAct Agent (Reason + Act Loop)

The `ReActAgent` below follows the ReAct pattern: at each step it **reasons** about what to do, **acts** by calling a tool, and **observes** the result before deciding on the next step. This is a simplified, rule-based version of the reasoning an LLM-based agent performs at each turn.

In [3]:
class ReActAgent:
    def __init__(self, name, tools, memory):
        self.name = name
        self.tools = tools
        self.memory = memory

    def run(self, goal, steps):
        """
        `steps` is a small scripted plan of (thought, tool_name, tool_input) tuples.
        In a real LLM-based agent, the model itself would generate these dynamically;
        here we simulate that reasoning explicitly so the loop is easy to follow.
        """
        print(f"[{self.name}] Goal: {goal}\n")
        for i, (thought, tool_name, tool_input) in enumerate(steps, 1):
            print(f"Step {i}")
            print(f"  Thought:     {thought}")
            action_desc = f"Call `{tool_name}` with input: {tool_input!r}"
            print(f"  Action:      {action_desc}")

            observation = self.tools[tool_name].use(tool_input)
            print(f"  Observation: {observation}\n")

            self.memory.remember({
                "step": i, "thought": thought, "tool": tool_name,
                "input": tool_input, "observation": observation
            })

        print(f"[{self.name}] Goal complete.")


agent_memory = Memory()
react_agent = ReActAgent("ResearchAgent", tools, agent_memory)

plan = [
    ("I need to find the capital of France.", "search", "capital of france"),
    ("I should also check the speed of light for the report.", "search", "speed of light"),
    ("The report needs the result of 12 times 8 as well.", "calculator", "12 * 8"),
]

react_agent.run("Prepare a short fact sheet", plan)

[ResearchAgent] Goal: Prepare a short fact sheet

Step 1
  Thought:     I need to find the capital of France.
  Action:      Call `search` with input: 'capital of france'
  Observation: Paris

Step 2
  Thought:     I should also check the speed of light for the report.
  Action:      Call `search` with input: 'speed of light'
  Observation: 299,792 km/s

Step 3
  Thought:     The report needs the result of 12 times 8 as well.
  Action:      Call `calculator` with input: '12 * 8'
  Observation: 96

[ResearchAgent] Goal complete.


### Interpretation

The ReAct loop shows each reasoning step (**Thought**), the resulting action (**Action**), and the tool's result (**Observation**). Chaining these steps together allows the agent to complete a multi-part goal using only the tools available to it, rather than a single fixed instruction.

## 3. Reviewing Agent Memory

After running the ReAct loop, the agent's memory contains a record of every step it took. This short-term memory can be reviewed, reused in later reasoning, or persisted for future sessions.

In [4]:
print("Agent memory log:\n")
agent_memory.show()

Agent memory log:

1. {'step': 1, 'thought': 'I need to find the capital of France.', 'tool': 'search', 'input': 'capital of france', 'observation': 'Paris'}
2. {'step': 2, 'thought': 'I should also check the speed of light for the report.', 'tool': 'search', 'input': 'speed of light', 'observation': '299,792 km/s'}
3. {'step': 3, 'thought': 'The report needs the result of 12 times 8 as well.', 'tool': 'calculator', 'input': '12 * 8', 'observation': 96}


### Interpretation

Each memory entry captures the thought, the tool used, the input given, and the observation received. Reviewing this log makes the agent's decision process transparent and auditable — an important property for trustworthy autonomous systems.

## 4. Planning Agent

A **Planning Agent** decomposes a broad goal into an ordered list of sub-tasks before execution, rather than deciding one step at a time. The function below simulates a planning step by breaking a high-level goal into smaller, actionable sub-tasks.

In [5]:
def plan_goal(goal):
    """A simple rule-based planner that breaks a goal into sub-tasks."""
    plans = {
        "prepare fact sheet": [
            "Look up the capital of France",
            "Look up the speed of light",
            "Calculate 12 x 8 for the appendix",
            "Summarize findings into a short report",
        ],
        "onboard new employee": [
            "Create employee record in the system",
            "Assign onboarding training modules",
            "Schedule a welcome meeting with the manager",
            "Send workspace access credentials",
        ],
    }
    return plans.get(goal.lower(), ["No predefined plan found for this goal."])

goal = "prepare fact sheet"
sub_tasks = plan_goal(goal)

print(f"Goal: {goal}\n")
print("Generated plan:")
for i, task in enumerate(sub_tasks, 1):
    print(f"  {i}. {task}")

Goal: prepare fact sheet

Generated plan:
  1. Look up the capital of France
  2. Look up the speed of light
  3. Calculate 12 x 8 for the appendix
  4. Summarize findings into a short report


### Interpretation

The planner turns one broad goal into a clear, ordered checklist. In a full agent system, each sub-task would then be executed — using the ReAct loop shown earlier — and the plan could be revised if a sub-task fails or new information is discovered.

## 5. Tool Calling / Function Calling

The example below demonstrates **function calling**: the agent is given a natural-language request, matches it to the correct registered tool by name, and executes that tool with the appropriate input — the same basic mechanism used when an LLM agent selects and calls a real API or function.

In [6]:
def call_tool_by_request(request_tool_name, request_input):
    if request_tool_name not in tools:
        return f"Error: no tool named '{request_tool_name}' is available."
    return tools[request_tool_name].use(request_input)

requests = [
    ("search", "largest planet"),
    ("calculator", "(15 + 5) * 3"),
    ("translate", "hello"),  # not a registered tool, to show error handling
]

for tool_name, tool_input in requests:
    result = call_tool_by_request(tool_name, tool_input)
    print(f"Requested tool: {tool_name:<10} Input: {tool_input!r:<20} -> Result: {result}")

Requested tool: search     Input: 'largest planet'     -> Result: Jupiter
Requested tool: calculator Input: '(15 + 5) * 3'       -> Result: 60
Requested tool: translate  Input: 'hello'              -> Result: Error: no tool named 'translate' is available.


### Interpretation

The third request asks for a tool (`translate`) that was never registered. The agent handles this gracefully by returning an error message rather than failing silently — illustrating why validating tool availability and inputs matters in real agent systems.

## 6. Multi-Agent System (Manager + Workers)

This section simulates a simple **multi-agent system**: a `ManagerAgent` receives a goal, breaks it into sub-tasks using the planner, and delegates each sub-task to a specialized `WorkerAgent`. This mirrors the manager–worker pattern commonly used to coordinate multiple agents on a complex task.

In [7]:
class WorkerAgent:
    def __init__(self, name, specialty):
        self.name = name
        self.specialty = specialty

    def handle_task(self, task):
        return f"[{self.name} - {self.specialty}] Completed: '{task}'"


class ManagerAgent:
    def __init__(self, workers):
        self.workers = workers  # dict of specialty -> WorkerAgent

    def assign_worker(self, task):
        task_lower = task.lower()
        if "look up" in task_lower:
            return self.workers["research"]
        if "calculate" in task_lower:
            return self.workers["math"]
        return self.workers["general"]

    def run(self, goal):
        print(f"[Manager] Received goal: {goal}\n")
        sub_tasks = plan_goal(goal)
        results = []
        for task in sub_tasks:
            worker = self.assign_worker(task)
            result = worker.handle_task(task)
            print(result)
            results.append(result)
        print("\n[Manager] All sub-tasks completed. Compiling final result...")
        return results


workers = {
    "research": WorkerAgent("Agent-R", "Research"),
    "math": WorkerAgent("Agent-M", "Math"),
    "general": WorkerAgent("Agent-G", "General"),
}

manager = ManagerAgent(workers)
final_results = manager.run("prepare fact sheet")

[Manager] Received goal: prepare fact sheet

[Agent-R - Research] Completed: 'Look up the capital of France'
[Agent-R - Research] Completed: 'Look up the speed of light'
[Agent-M - Math] Completed: 'Calculate 12 x 8 for the appendix'
[Agent-G - General] Completed: 'Summarize findings into a short report'

[Manager] All sub-tasks completed. Compiling final result...


### Interpretation

The `ManagerAgent` never performs the sub-tasks itself — it plans, assigns, and coordinates. Each `WorkerAgent` handles only the sub-tasks that match its specialty (research, math, or general). This division of labor is the core idea behind multi-agent systems: complex goals are handled more reliably by combining several focused agents than by relying on one agent to do everything.

## 7. Workforce-Style Summary of the Run

To mirror a typical analytics summary, the cell below aggregates simple statistics about the multi-agent run: how many sub-tasks were completed and how many were handled by each worker.

In [8]:
from collections import Counter

worker_counts = Counter()
for task in plan_goal("prepare fact sheet"):
    worker = manager.assign_worker(task)
    worker_counts[worker.name] += 1

print("Task distribution across workers:")
for worker_name, count in worker_counts.items():
    print(f"  {worker_name}: {count} task(s)")

print(f"\nTotal sub-tasks completed: {sum(worker_counts.values())}")

Task distribution across workers:
  Agent-R: 2 task(s)
  Agent-M: 1 task(s)
  Agent-G: 1 task(s)

Total sub-tasks completed: 4


### Interpretation

This simple distribution shows how work was divided among agents. In a production multi-agent system, similar metrics (task counts, success rates, latency per agent) would help identify bottlenecks or imbalanced workloads across the agent team.

## 8. Conclusion

This notebook demonstrated how core autonomous-agent concepts can be implemented in simple, beginner-friendly Python: a ReAct-style reasoning loop, short-term memory, goal planning, tool/function calling, and a manager–worker multi-agent system.

These lightweight simulations illustrate the underlying mechanics that power real LLM-based agents, which replace the scripted "thoughts" and rule-based planner used here with a language model that reasons and decides dynamically at each step.